# Qwen3-VL + UMLS concept guidance on ROCOv2 &mdash; **Model B**

Notebook counterpart of `qwen3vl_roco_finetune_cui.py` (run the `.py` under `tmux` for the
real thing; use this to inspect the setup and run the **probe**).

**Model B = Model A + an auxiliary multi-label concept head on the merger output.**
Data subset, seed, resolution, LoRA config, learning rates, schedule, prompt and decoding
are all identical to `qwen3vl_roco_finetune.py`, so the **only** difference is the CUI
guidance. Setting `LAMBDA_CUI = 0` reproduces Model A exactly, through this same code path.

### Precedent
[AI Stat Lab, ImageCLEFmedical Caption 2025](https://ceur-ws.org/Vol-4038/paper_196.pdf)
(3rd of 8) mean-pool their **Q-Former** output and attach two linear classifiers
(2,478 CUIs + 21 semantic types), training with
$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{caption}} + \lambda\,\mathcal{L}_{\text{cls}}$
at $\lambda = 0.1$. Their Table 2 ablation (dual encoder, #1673 &rarr; #1695) is the only
clean isolation of this variable in the literature: **ROUGE-1 +0.0062, UMLS F1 +0.0048,
AlignScore &minus;0.0053**. Our merger occupies the identical structural slot as their
Q-Former (the trained vision&rarr;LLM projector).

### Two documented deviations
1. **BCE with `pos_weight`** instead of multi-label margin loss. BCE is what every top
   ImageCLEF-2025 concept-detection team used (AUEB, DeepLens, UIT-Oggy); `pos_weight`
   handles the ~3-of-1571 imbalance explicitly, and we never need calibrated probabilities
   because the head is discarded at inference. `CUI_LOSS = "margin"` reproduces their choice.
2. **Coarse head = ROCOv2's 18 manually curated concepts** (modality / body region /
   directionality) rather than 21 UMLS semantic types, which ROCOv2 does not ship. Same
   role (small, dense, low-tail) and hand-curated rather than auto-extracted.

### Why this framing is clean
The head sits **before** the LLM and the ViT is frozen, so the auxiliary gradient reaches
**only the merger**. The earlier LLaVA result was about connector *bandwidth*
(1 &rarr; 196 visual tokens); this is connector *content*. Same component, orthogonal axis.
And because nothing is consumed at inference, this avoids the error propagation from an
imperfect CUI detector that DS4DH identified as what sank the concept-injection methods.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import json, math, random, time, collections
import torch
import torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if device == "cuda" else torch.float32
print("device =", device, "| dtype =", DTYPE)

In [ ]:
# ------------------------------- Config ------------------------------- #
MODEL_ID     = "/home/matei/qwen3-vl-4b-instruct"
RUN_TAG      = "qwen3vl_ft_cui"
OUT_DIR      = f"/home/matei/{RUN_TAG}_ckpt"

# --- identical to Model A ---
N_TRAIN, N_VAL, SEED = 12000, 500, 42          # same seed -> same 12k subset as Model A
MIN_PIXELS   = 256 * 32 * 32
MAX_PIXELS   = 768 * 32 * 32
MAX_CAP_TOK  = 48
EPOCHS       = 2
BATCH_SIZE   = 1
GRAD_ACCUM   = 4
LR_LORA      = 1e-4
LR_MERGER    = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRAC  = 0.03
MAX_GRAD_NORM= 1.0
TRAIN_MERGER = True
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EVAL_EVERY, PATIENCE = 500, 3
PROBE_STEPS  = 20

# --- the CUI guidance: the ONLY block that differs from Model A ---
LAMBDA_CUI   = 0.1        # 0 -> exactly Model A
MIN_CUI_FREQ = 10         # ImageCLEF's own curation rule -> 1,571 classes
USE_MANUAL   = True       # the 18-class curated head
CUI_LOSS     = "bce"      # "bce" | "margin"
POSW_CLAMP   = 50.0       # cap pos_weight so ultra-rare classes don't dominate
LR_HEAD      = 1e-4       # heads are randomly initialised -> LoRA-scale LR

In [ ]:
# ---------------- The V1 prompt (identical to Model A and to the eval script) ---------------- #
SYSTEM_PROMPT = (
    "You are an expert radiologist writing captions for a radiology teaching archive. "
    "You are given one medical image. Write a single concise caption that describes only "
    "what is directly visible.\n"
    "Rules:\n"
    "1. Describe only what is observable in this image: the imaging modality, the anatomical "
    "region, and any clearly visible findings. Do not infer diagnoses, patient history, or "
    "findings that are not directly visible.\n"
    "2. If a finding cannot be determined from the image, do not state it.\n"
    "3. Be terse and clinical: one sentence, radiology-report style.\n"
    "4. Output only the caption -- no preamble, no \"This image shows\", no disclaimers."
)
USER_PROMPT = "Provide the caption for this image."

In [ ]:
# ---------------------- Load model + processor ---------------------- #
from transformers import AutoProcessor, AutoModelForImageTextToText, get_cosine_schedule_with_warmup

processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "right"
MERGE_SIZE = getattr(processor.image_processor, "merge_size", 2)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map={"": 0} if device == "cuda" else None)
model.config.use_cache = False

# Grab the merger BEFORE the PEFT wrapper: the same module object stays in the graph, so a
# forward hook on it still fires and its output remains connected to autograd.
merger_module = model.model.visual.merger
D_MERGER = merger_module.linear_fc2.out_features        # == LLM hidden size
print("merger output dim:", D_MERGER)

In [ ]:
# ------------- Freeze everything, then re-enable LoRA + the mergers ------------- #
for p in model.parameters():
    p.requires_grad = False

MERGER_PREFIXES = ("model.visual.merger", "model.visual.deepstack_merger_list")
merger_params = []
if TRAIN_MERGER:
    for n, p in model.named_parameters():
        if n.startswith(MERGER_PREFIXES):
            p.requires_grad = True
            p.data = p.data.float()
            merger_params.append(p)

from peft import LoraConfig, get_peft_model
lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_cfg)
# PEFT's _mark_only_adapters_as_trainable() re-freezes EVERY non-LoRA parameter,
# including the mergers unfrozen above. Re-assert them -- merger_params holds the same
# Parameter objects, so this restores the flags on the modules actually in the graph.
for p in merger_params:
    p.requires_grad = True
assert all(p.requires_grad for p in merger_params), "merger re-freeze not repaired"
for n, p in model.named_parameters():
    if "lora_" in n:
        p.data = p.data.float(); p.requires_grad = True

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
lora_params = [p for n, p in model.named_parameters() if "lora_" in n and p.requires_grad]
print(f"VRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ------------------- Label spaces, built from TRAIN only ------------------- #
import pandas as pd
ROCO_DIR = "/home/matei/rocov2"

def read_concepts(fname):
    df = pd.read_csv(os.path.join(ROCO_DIR, fname))
    return {r.ID: [c for c in str(r.CUIs).split(";") if c and c != "nan"]
            for r in df.itertuples()}

cui_train, cui_valid = read_concepts("train_concepts.csv"), read_concepts("valid_concepts.csv")
man_train = read_concepts("train_concepts_manual.csv") if USE_MANUAL else {}
man_valid = read_concepts("valid_concepts_manual.csv") if USE_MANUAL else {}

_freq = collections.Counter(c for v in cui_train.values() for c in v)
CUI_VOCAB = sorted([c for c, n in _freq.items() if n >= MIN_CUI_FREQ])
CUI2IDX   = {c: i for i, c in enumerate(CUI_VOCAB)}
MAN_VOCAB = sorted({c for v in man_train.values() for c in v}) if USE_MANUAL else []
MAN2IDX   = {c: i for i, c in enumerate(MAN_VOCAB)}
N_CUI, N_MAN = len(CUI_VOCAB), len(MAN_VOCAB)
print(f"{N_CUI} CUIs (>= {MIN_CUI_FREQ} occurrences) | {N_MAN} curated concepts")

names = pd.read_csv(os.path.join(ROCO_DIR, "cui_mapping.csv")).set_index("CUI")["Canonical name"].to_dict()
print("curated:", [names.get(c, c) for c in MAN_VOCAB])

In [ ]:
# ------------------- Auxiliary heads: the intervention ------------------- #
# Mean-pool the merger's visual tokens -> two linear classifiers, exactly as AI Stat Lab do
# on their Q-Former output. fp32 so AdamW updates stay stable.
def pos_weight_for(vocab, idx, id2list, n_images):
    cnt = collections.Counter(c for v in id2list.values() for c in v if c in idx)
    w = torch.ones(len(vocab))
    for c, i in idx.items():
        p = max(1, cnt.get(c, 0))
        w[i] = min((n_images - p) / p, POSW_CLAMP)
    return w

head_cui = nn.Linear(D_MERGER, N_CUI).to(device, torch.float32)
head_man = nn.Linear(D_MERGER, N_MAN).to(device, torch.float32) if N_MAN else None
head_params = list(head_cui.parameters()) + (list(head_man.parameters()) if head_man else [])

_pw_cui = pos_weight_for(CUI_VOCAB, CUI2IDX, cui_train, len(cui_train)).to(device)
_pw_man = pos_weight_for(MAN_VOCAB, MAN2IDX, man_train, len(man_train)).to(device) if head_man else None
margin_loss = nn.MultiLabelMarginLoss()

# Forward hook captures the merger output. Cleared before every forward and read immediately
# after, so the recomputation gradient checkpointing triggers during backward can't overwrite it.
_captured = []
merger_module.register_forward_hook(lambda m, i, o: _captured.append(o))

def pooled_visual(grid_thw):
    out = _captured[0]                                  # [total_merged_tokens, D]
    if out.dim() == 3:
        return out.mean(dim=1)
    per_img = (grid_thw.prod(dim=-1) // (MERGE_SIZE ** 2)).tolist()
    return torch.stack([c.mean(0) for c in torch.split(out, per_img, dim=0)])

def concept_targets(ids, n, idx, id2list):
    y, m = torch.zeros(len(ids), n), torch.zeros(len(ids))
    for b, _id in enumerate(ids):
        got = [idx[c] for c in id2list.get(_id, []) if c in idx]
        if got:
            y[b, got] = 1.0; m[b] = 1.0                 # 493 train images have no curated concept
    return y.to(device), m.to(device)

def aux_loss(pooled, ids, split_cui, split_man):
    parts, p32 = {}, pooled.float()
    y, m = concept_targets(ids, N_CUI, CUI2IDX, split_cui)
    if m.sum() > 0:
        logit = head_cui(p32)
        if CUI_LOSS == "margin":
            tgt = torch.full_like(logit, -1, dtype=torch.long)
            for b in range(len(ids)):
                pos = y[b].nonzero(as_tuple=True)[0]
                tgt[b, :len(pos)] = pos
            parts["cui"] = margin_loss(logit, tgt)
        else:
            per = nn.functional.binary_cross_entropy_with_logits(
                logit, y, pos_weight=_pw_cui, reduction="none").mean(dim=1)
            parts["cui"] = (per * m).sum() / m.sum()
    if head_man is not None:
        y2, m2 = concept_targets(ids, N_MAN, MAN2IDX, split_man)
        if m2.sum() > 0:
            per = nn.functional.binary_cross_entropy_with_logits(
                head_man(p32), y2, pos_weight=_pw_man, reduction="none").mean(dim=1)
            parts["man"] = (per * m2).sum() / m2.sum()
    return parts

In [ ]:
# ------------------ Data: same loader, same seed, same subset as Model A ------------------ #
def load_split(split, csv):
    img_dir = os.path.join(ROCO_DIR, split)
    caps = pd.read_csv(os.path.join(ROCO_DIR, csv)).dropna(subset=["Caption"]).reset_index(drop=True)
    return [{"id": r.ID, "path": os.path.join(img_dir, f"{r.ID}.jpg"), "caption": str(r.Caption)}
            for r in caps.itertuples() if os.path.isfile(os.path.join(img_dir, f"{r.ID}.jpg"))]

train_all, val_all = load_split("train", "train_captions.csv"), load_split("valid", "valid_captions.csv")
rng = random.Random(SEED); rng.shuffle(train_all); rng.shuffle(val_all)
train_recs, val_recs = train_all[:N_TRAIN], val_all[:N_VAL]
print(f"train {len(train_recs)} | val {len(val_recs)}")

_DUMMY = Image.new("RGB", (32, 32))
_msgs = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user",   "content": [{"type": "image", "image": _DUMMY},
                                   {"type": "text",  "text": USER_PROMPT}]},
]
PROMPT_TEXT = processor.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)
EOS = processor.tokenizer.eos_token or "<|im_end|>"

def build_example(rec):
    """Prompt+caption tokenised; labels = -100 everywhere except the caption tokens."""
    img = Image.open(rec["path"]).convert("RGB")
    cap = " ".join(str(rec["caption"]).split())
    cap_ids = processor.tokenizer(cap, add_special_tokens=False)["input_ids"][:MAX_CAP_TOK]
    cap = processor.tokenizer.decode(cap_ids)
    full = processor(text=[PROMPT_TEXT + cap + EOS], images=[img], return_tensors="pt")
    plen = processor(text=[PROMPT_TEXT], images=[img], return_tensors="pt")["input_ids"].shape[1]
    labels = full["input_ids"].clone()
    labels[:, :plen] = -100
    labels[full["attention_mask"] == 0] = -100
    full["labels"] = labels
    return full

# Sanity: only the caption is supervised, and the concept labels line up.
ex = build_example(train_recs[0]); lab = ex["labels"][0]
print(f"seq={lab.numel()} supervised={(lab != -100).sum().item()}")
print("supervised text ->", processor.tokenizer.decode(lab[lab != -100])[:80])
r = train_recs[0]
print(f"{r['id']}: {len(cui_train.get(r['id'],[]))} CUIs | curated ->",
      [names.get(c, c) for c in man_train.get(r["id"], [])])

In [ ]:
# ------------------- Optimiser: LoRA / merger / heads ------------------- #
groups = [{"params": lora_params, "lr": LR_LORA, "weight_decay": WEIGHT_DECAY}]
if merger_params:
    groups.append({"params": merger_params, "lr": LR_MERGER, "weight_decay": WEIGHT_DECAY})
if LAMBDA_CUI > 0 and head_params:
    groups.append({"params": head_params, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY})
optim = torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8)

n_lora, n_merger, n_head = (sum(p.numel() for p in x)
                            for x in (lora_params, merger_params, head_params))
print(f"trainable: LoRA {n_lora/1e6:.1f}M + merger {n_merger/1e6:.1f}M + heads {n_head/1e6:.1f}M")

steps_per_epoch = max(1, math.ceil(len(train_recs) / (BATCH_SIZE * GRAD_ACCUM)))
total_steps     = max(1, int(steps_per_epoch * EPOCHS))
sched = get_cosine_schedule_with_warmup(optim, int(WARMUP_FRAC * total_steps), total_steps)
print(f"{steps_per_epoch} steps/epoch x {EPOCHS} epochs = {total_steps} total")

def forward_losses(rec, split_cui, split_man):
    ex = {k: v.to(model.device) for k, v in build_example(rec).items()}
    _captured.clear()
    with torch.autocast("cuda", dtype=DTYPE):
        out = model(**ex)
    l_cap, parts = out.loss, {}
    if LAMBDA_CUI > 0 and _captured:
        parts = aux_loss(pooled_visual(ex["image_grid_thw"]), [rec["id"]], split_cui, split_man)
    total = l_cap + LAMBDA_CUI * sum(parts.values()) if parts else l_cap
    return total, l_cap, parts

In [ ]:
# ------------- PROBE: peak VRAM, s/step, and the lambda scale check ------------- #
# Run this BEFORE committing days of GPU. Compare peak VRAM against the Model A probe:
# the heads should add only ~0.1 GB. If it OOMs, apply the SAME fix to Model A too.
model.train()
torch.cuda.reset_peak_memory_stats()
t0, last = time.time(), {}
for i in tqdm(range(PROBE_STEPS), desc="probe"):
    total, l_cap, parts = forward_losses(train_recs[i], cui_train, man_train)
    (total / GRAD_ACCUM).backward()
    last = {"caption": l_cap.item(), **{f"aux_{k}": v.item() for k, v in parts.items()}}
    if (i + 1) % GRAD_ACCUM == 0:
        # Clip the caption path SEPARATELY from the heads: a joint clip folds the heads'
        # large random-init gradient norm into the global norm and scales the caption path
        # down too -- an effective LR cut Model A never experiences.
        n_cap = torch.nn.utils.clip_grad_norm_(lora_params + merger_params, MAX_GRAD_NORM)
        n_head = (torch.nn.utils.clip_grad_norm_(head_params, MAX_GRAD_NORM)
                  if head_params else torch.tensor(0.))
        if i + 1 == GRAD_ACCUM:
            print(f"\n[grad norms] caption {float(n_cap):.2f} | heads {float(n_head):.2f}")
        optim.step(); sched.step(); optim.zero_grad(set_to_none=True)

dt, peak = (time.time() - t0) / PROBE_STEPS, torch.cuda.max_memory_allocated() / 1e9
aux_sum = sum(v for k, v in last.items() if k.startswith("aux_"))
print(f"\nMAX_VIS_TOKENS : {MAX_PIXELS // (32*32)} | lambda {LAMBDA_CUI} | loss {CUI_LOSS}")
print(f"peak VRAM      : {peak:.2f} GB / 12.6 GB")
print(f"s/sample       : {dt:.2f}  ->  1 epoch = {dt*len(train_recs)/3600:.1f} h")
print(f"losses         : {last}")
if aux_sum:
    print(f"weighted aux / caption = {LAMBDA_CUI*aux_sum/last['caption']:.1%} "
          f"(want roughly 5-20%; tune LAMBDA_CUI if far off)")

### Next: run headless, then evaluate

```bash
cd /home/matei
# probe BOTH models before running either, and lock one config that fits both
PROBE=1 miniconda3/envs/vlm/bin/python qwen3vl_roco_finetune.py         # Model A
PROBE=1 miniconda3/envs/vlm/bin/python qwen3vl_roco_finetune_cui.py     # Model B

tmux new -d -s qwen_ft_cui \
  'cd /home/matei && miniconda3/envs/vlm/bin/python qwen3vl_roco_finetune_cui.py 2>&1 | tee ft_cui_run.log'
```

**Evaluation needs no change at all** &mdash; the concept heads are dropped at inference, so
Model B is scored by the same script, with the same prompt and decoding, as the zero-shot
and Model A runs:

```bash
LORA_PATH=/home/matei/qwen3vl_ft_cui_ckpt/best RUN_TAG=qwen3vl_ft_cui_test \
  miniconda3/envs/vlm/bin/python qwen3vl_roco_zeroshot_v1.py
```

**Reference points:** zero-shot V1 = **0.6539** BERTScore; best existing system
(LLaVA ViT&minus;1) = **0.6708**. And note the honest expectation: AI Stat Lab's isolated
gain was small (+0.0062 ROUGE-1, +0.0048 UMLS F1) on 80k images and 10 epochs &mdash; on a
12k subset a null result is a realistic and perfectly reportable outcome. It also lands
mostly on **factuality**, which the current relevance-only metric suite cannot see, so
UMLS Concept-F1 is needed before this comparison can be properly judged.